# Voice Cloning Test Sample Generation (Coqui XTTS-v2)
### Audio Deepfake Detection Project — Auxiliary Test Set Module

> **Ethical Notice & Scope:**
> - This module is used strictly to generate a small controlled test set (~15-20 clips) from an **internal, consenting team member's voice**.
> - **Pretrained XTTS-v2 Model:** Developed by Coqui AI (pretrained foundation model, not our work).
> - **Our Work:** Test set design, audio preprocessing integration, spectral feature extraction, and detector evaluation.

## Step 1: Environment Verification & Dependency Installation

In [ ]:
# 1. Verify GPU Availability and VRAM (T4 GPU should be visible)
!nvidia-smi

In [ ]:
# 2. Install Coqui TTS and apply the automated compatibility patch for Transformers 5+
!pip install -q coqui-tts soundfile

# Automated patch for transformers.pytorch_utils isin_mps_friendly deprecation
import glob
pattern = "/usr/local/lib/python*/dist-packages/TTS/tts/layers/tortoise/autoregressive.py"
target_files = glob.glob(pattern)
for filepath in target_files:
    with open(filepath, "r") as f:
        content = f.read()
    old_stmt = "from transformers.pytorch_utils import isin_mps_friendly as isin"
    new_stmt = "import torch\nisin = torch.isin"
    if old_stmt in content:
        content = content.replace(old_stmt, new_stmt)
        with open(filepath, "w") as f:
            f.write(content)
        print(f"[SUCCESS] Patched compatibility in:\n  {filepath}")
    else:
        print("[OK] File already patched or up-to-date.")

In [ ]:
# 3. Verify PyTorch, CUDA, and TTS Imports
import torch
import torchaudio
from TTS.api import TTS

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Allocated GPU   : {gpu_name}")
    print(f"Available VRAM  : {vram_gb:.2f} GB")
    print("\n[SUCCESS] Environment verified and ready for Step 2!")
else:
    print("WARNING: GPU still not detected.")

## Step 2: Single Working Example (Few-Shot Voice Cloning)

**Objective:**
1. Load pretrained XTTS-v2 onto the GPU.
2. Upload ONE consented reference recording from our team member (~6-10 seconds WAV).
3. Synthesize ONE cloned test sample using a short text prompt.
4. Listen to the generated sample to verify acoustic naturalness.

In [ ]:
# 4. Load Pretrained XTTS-v2 Model onto GPU
# Downloads model weights (~3.5 GB) on first execution
print("Loading XTTS-v2 model onto GPU...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
print("[OK] XTTS-v2 model loaded successfully!")

In [ ]:
# 5. Upload Consented Reference Audio File
# Select a 6-10 second WAV file recorded by your team member
from google.colab import files
import os, shutil

print("Please upload your team member's reference recording (WAV format, ~6-10 seconds):")
uploaded = files.upload()

if uploaded:
    raw_filename = list(uploaded.keys())[0]
    ref_path = "reference_teammate.wav"
    shutil.move(raw_filename, ref_path)
    print(f"\n[OK] Reference audio saved as '{ref_path}'")
else:
    raise FileNotFoundError("No reference audio file was uploaded. Please provide your consented recording.")

In [ ]:
# 6. Synthesize Single Cloned Audio Sample
test_prompt = "Hello, this is an AI-generated clone of my voice created for our audio deepfake detection project."
output_wav = "sample_clone_001.wav"

print(f"Synthesizing speech using reference '{ref_path}'...")
print(f"Prompt: \"{test_prompt}\"")

tts.tts_to_file(
    text=test_prompt,
    speaker_wav=ref_path,
    language="en",
    file_path=output_wav
)

print(f"\n[SUCCESS] Cloned sample generated: {output_wav}")

In [ ]:
# 7. Audio Playback & Comparison
import IPython.display as ipd

print("--- ORIGINAL REFERENCE AUDIO (Team Member) ---")
display(ipd.Audio("reference_teammate.wav"))

print("\n--- AI-GENERATED CLONE (XTTS-v2) ---")
display(ipd.Audio("sample_clone_001.wav"))